# Table 5 (protein): Lineage-Holdout vs. Random-Split Significance Test

Tests whether each architecture's mean AUC under lineage holdout differs significantly from its
Table 3 (`tab:combined_auc_summary`) random-split test AUC, per architecture, paired across the 9
eligible drugs, with a Holm-Bonferroni correction across the 4 architectures.

Baseline (Table 3, protein) is read from `Combined2_test_aucs_Table.csv`, which as of 2026-09-21 uses
`regression_all_results_metrics.csv` for the LogReg (Ref-Alt) column -- this replaced an
unreproducible earlier LogReg source that did not match its own generating script when rerun.

In [1]:
import pandas as pd, numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

DRUGS = ["rifampicin","isoniazid","ethambutol","pyrazinamide","streptomycin",
         "moxifloxacin","ethionamide","amikacin","capreomycin"]

LINEAGE_SOURCES = {
    "LogReg":      ("data/latest/lineage_ood_all_train/regression/{drug}/all_lineage_summary.csv", "logreg"),
    "CNN":         ("data/latest/lineage_ood_all_train/cnn/{drug}/all_lineage_summary.csv", None),
    "Transformer": ("data/latest/lineage_ood_all_train/transformer/{drug}/all_lineage_summary.csv", None),
    "ESM-320":     ("data/latest/lineage_ood_all_train/esm/{drug}/full_320/all_lineage_summary.csv", None),
}

RANDOM_SPLIT_COLS = {
    "LogReg": "logreg_test_auc", "CNN": "cnn_test_auc",
    "Transformer": "transformer_test_auc", "ESM-320": "esm_full320_test_auc",
}

def lineage_mean(path, model_filter=None):
    d = pd.read_csv(path)
    d = d[d["feasible"] == True]
    if model_filter:
        d = d[d["model"] == model_filter]
    return d["auc"].mean()

In [2]:
table3 = pd.read_csv("data/latest/results/prediction/combined/Combined2_test_aucs_Table.csv").set_index("drug")

lineage = {}
for arch, (path_tpl, model_filter) in LINEAGE_SOURCES.items():
    lineage[arch] = {drug: lineage_mean(path_tpl.format(drug=drug), model_filter) for drug in DRUGS}

rows = []
for arch in LINEAGE_SOURCES:
    a = np.array([lineage[arch][d] for d in DRUGS])
    b = np.array([table3.loc[d, RANDOM_SPLIT_COLS[arch]] for d in DRUGS])
    delta = a - b
    _, p = wilcoxon(a, b, alternative="two-sided")
    rows.append({
        "architecture": arch,
        "mean_random_split_auc": b.mean(),
        "mean_lineage_holdout_auc": a.mean(),
        "mean_delta_auc": delta.mean(),
        "p_value": p,
        "n_drugs": len(DRUGS),
    })

result = pd.DataFrame(rows)
result["p_value_holm"] = multipletests(result["p_value"], method="holm")[1]
result.to_csv("data/latest/lineage_ood_all_train/table5_lineage_vs_random_significance_holm.csv", index=False)
print("[OK] Wrote table5_lineage_vs_random_significance_holm.csv")
result.round(4)

[OK] Wrote table5_lineage_vs_random_significance_holm.csv


,architecture,mean_random_split_auc,mean_lineage_holdout_auc,mean_delta_auc,p_value,n_drugs,p_value_holm
0,LogReg,0.7877,0.7504,-0.0373,0.0039,9,0.0156
1,CNN,0.7659,0.7467,-0.0193,0.0039,9,0.0156
2,Transformer,0.6137,0.6021,-0.0116,0.7344,9,0.7344
3,ESM-320,0.7541,0.7245,-0.0296,0.0742,9,0.1484
